In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Exploratory Analysis — VAE Hybrid Language Music Clustering\n",
        "\n",
        "This notebook provides quick checks and exploratory analysis for:\n",
        "- Lyrics-only results (Easy + Medium)\n",
        "- Audio-only results\n",
        "- Fused (audio + lyrics) β-VAE results\n",
        "\n",
        "**Note:** Datasets are gitignored. This notebook focuses on results CSVs and saved figures."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import os\n",
        "from pathlib import Path\n",
        "import pandas as pd\n",
        "\n",
        "PROJECT_ROOT = Path('..')  # notebooks/ -> project/\n",
        "RESULTS_DIR = PROJECT_ROOT / 'results'\n",
        "VIS_DIR = RESULTS_DIR / 'latent_visualization'\n",
        "\n",
        "print('Project root:', PROJECT_ROOT.resolve())\n",
        "print('Results dir:', RESULTS_DIR.resolve())\n",
        "print('Visualization dir:', VIS_DIR.resolve())"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1) Load Metrics Files\n",
        "\n",
        "This notebook will try to load common output CSVs. If a file is missing, it will skip it."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "metric_files = [\n",
        "    RESULTS_DIR / 'clustering_metrics.csv',\n",
        "    RESULTS_DIR / 'clustering_metrics_medium_lyrics.csv',\n",
        "    RESULTS_DIR / 'clustering_metrics_audio_only.csv',\n",
        "    RESULTS_DIR / 'clustering_metrics_fused_betaVAE.csv',\n",
        "]\n",
        "\n",
        "dfs = {}\n",
        "for f in metric_files:\n",
        "    if f.exists():\n",
        "        dfs[f.name] = pd.read_csv(f)\n",
        "        print(f'Loaded {f.name}: shape={dfs[f.name].shape}')\n",
        "    else:\n",
        "        print(f'Skipped (missing): {f.name}')\n",
        "\n",
        "list(dfs.keys())"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2) Quick View: Easy Task (Lyrics-only PCA vs VAE)\n",
        "\n",
        "Looks for `clustering_metrics.csv`."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "easy_name = 'clustering_metrics.csv'\n",
        "if easy_name in dfs:\n",
        "    df_easy = dfs[easy_name].copy()\n",
        "    display(df_easy)\n",
        "else:\n",
        "    print('Missing:', easy_name)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3) Medium Lyrics Results: Multiple Clusterers\n",
        "\n",
        "Shows which methods failed (e.g., DBSCAN collapsing to 1 cluster)."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "med_lyrics_name = 'clustering_metrics_medium_lyrics.csv'\n",
        "if med_lyrics_name in dfs:\n",
        "    df_ml = dfs[med_lyrics_name].copy()\n",
        "    display(df_ml)\n",
        "    \n",
        "    # show only KMeans/Agglomerative rows for clean comparison\n",
        "    filt = df_ml['clusterer'].str.contains('KMeans|Agglomerative', regex=True, na=False)\n",
        "    display(df_ml[filt].sort_values(['feature', 'clusterer']))\n",
        "else:\n",
        "    print('Missing:', med_lyrics_name)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4) Audio-only Results\n",
        "\n",
        "Print best result by Silhouette (among KMeans/Agglomerative) and show the full table."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "audio_name = 'clustering_metrics_audio_only.csv'\n",
        "if audio_name in dfs:\n",
        "    df_a = dfs[audio_name].copy()\n",
        "    display(df_a)\n",
        "    \n",
        "    filt = df_a['clusterer'].str.contains('KMeans|Agglomerative', regex=True, na=False)\n",
        "    df_main = df_a[filt].copy()\n",
        "    best = df_main.sort_values('silhouette', ascending=False).head(1)\n",
        "    print('Best audio-only (by silhouette):')\n",
        "    display(best)\n",
        "else:\n",
        "    print('Missing:', audio_name)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5) Fused + β-VAE Results (Hard Task)\n",
        "\n",
        "Find best β setting for clustering and show it."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "fused_name = 'clustering_metrics_fused_betaVAE.csv'\n",
        "if fused_name in dfs:\n",
        "    df_f = dfs[fused_name].copy()\n",
        "    display(df_f)\n",
        "    \n",
        "    # Best by CH (structure) among KMeans rows\n",
        "    km = df_f[df_f['clusterer'].str.contains('KMeans', na=False)].copy()\n",
        "    best_ch = km.sort_values('calinski_harabasz', ascending=False).head(1)\n",
        "    print('Best fused (KMeans) by Calinski–Harabasz:')\n",
        "    display(best_ch)\n",
        "    \n",
        "    # Best by Silhouette among KMeans rows\n",
        "    best_sil = km.sort_values('silhouette', ascending=False).head(1)\n",
        "    print('Best fused (KMeans) by Silhouette:')\n",
        "    display(best_sil)\n",
        "else:\n",
        "    print('Missing:', fused_name)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6) Check Visualization Files Exist\n",
        "\n",
        "This lists the expected UMAP figure filenames. If any are missing, the notebook will tell you."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "expected_figs = [\n",
        "    'umap_pca_kmeans.png',\n",
        "    'umap_vae_kmeans.png',\n",
        "    'umap_audio_pca16_kmeans.png',\n",
        "    'umap_audio_vae16_kmeans.png',\n",
        "    'umap_fused_pca16_kmeans.png',\n",
        "    'umap_fused_vae16_beta1_kmeans.png',\n",
        "    'umap_fused_vae16_beta4_kmeans.png',\n",
        "]\n",
        "\n",
        "missing = []\n",
        "for fn in expected_figs:\n",
        "    p = VIS_DIR / fn\n",
        "    if not p.exists():\n",
        "        missing.append(fn)\n",
        "\n",
        "print('Missing figures:' if missing else 'All expected figures found ✅')\n",
        "for m in missing:\n",
        "    print(' -', m)"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.x"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
